In [1]:
import string
from typing import Callable

import nltk
import pandas as pd
from afinn import Afinn
from nrclex import NRCLex
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import BaseEstimator
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import statsmodels.api as sm

nltk.download('punkt_tab')

# Initialize lexicon tools
vader = SentimentIntensityAnalyzer()
afinn = Afinn()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/ozercavdar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Import Data 

In [2]:
# import data 
url = "https://github.com/bozercavdar/kickstarter-project/releases/download/v1.0/kickstarter_data_OSF.csv"
local_path = "dataset.csv"

# UNCOMMENT THE LINE BELOW IF YOU DON'T HAVE THE DATASET IN THE LOCAL DIRECTORY
# df = pd.read_csv(url)
df = pd.read_csv(local_path)

In [3]:
# Filter only those required columns
selected_df = df[['uid', 'blurb', 'goal', 'state', 'usd_pledged', 'category']]
# Add column that states if a project is succesful or not
selected_df['success'] = selected_df['state'] == 'successful'
selected_df.head()

/tmp/ipykernel_1206/2469478530.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['success'] = selected_df['state'] == 'successful'


,uid,blurb,goal,state,usd_pledged,category,success
0,1,This project is designed to help protect the e...,2500.0,failed,0.0,Music,False
1,2,Help us built a sustainable studio & eliminate...,25000.0,failed,1.0,Technology,False
2,3,"""If I paint something, I don't want to have to...",5000.0,failed,5.0,Art,False
3,4,Our free app will allow you pool reservations ...,12000.0,failed,0.0,Food,False
4,5,Prohibition themed Gastro Pub and After Dark S...,20000.0,failed,0.0,Food,False


In [4]:
# Function to apply a feature extractor function on to a df
def extract_feature(df: pd.DataFrame, feature_extractor: Callable):
    df = df.join(df["blurb"].apply(feature_extractor))
    return df
    

In [5]:
# Function to fit a list of classifiers based on the given cols
def classifier(df: pd.DataFrame, cols: list[str], estimators: list[BaseEstimator]):
    X = df[cols].copy()
    y = df["success"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=41, stratify=y
    )

    total_results = None

    for estimator in estimators:
        estimator.fit(X_train, y_train)

        y_pred = estimator.predict(X_test)
        # y_pred_prob = estimator.predict_proba(X_test)[:, 1]

        results = pd.DataFrame({
            "Metric": ["success_pred_rate", "accuracy", "precision", "recall"],
            f"{estimator.__class__.__name__}": [
                y_pred.mean(),
                accuracy_score(y_test, y_pred),
                precision_score(y_test, y_pred),
                recall_score(y_test, y_pred)
                # roc_auc_score(y_test, y_pred_prob)
            ]
        })

        if total_results is None:
            total_results = results
        else:
            total_results = pd.merge(total_results, results, on="Metric")

    return total_results

In [ ]:
def logistic_regression_summary(df, col_names):
    X = df[col_names].copy()
    X = sm.add_constant(X) 
    y = df["success"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=41, stratify=y
    )

    model = sm.Logit(y_train, X_train)
    result = model.fit()

    print(result.summary())

In [7]:
# Randomly select 10000 projects to analyze
sample_df = selected_df.sample(10000, random_state=41)
sample_df

,uid,blurb,goal,state,usd_pledged,category,success
155006,189791,Spend less time searching your purse and more ...,20000.0,successful,23088.608880,Fashion,True
69020,77192,Kevin Jenkins will choreograph and direct a ne...,3000.0,successful,3635.318822,web,True
75028,83929,Finalist in the Telio National Design Competit...,800.0,successful,1126.748389,Fashion,True
19323,21429,What are the real stories behind these unique ...,5000.0,failed,450.000000,Journalism,False
22301,24722,"High cost-performance, Fastest, Highest accura...",50000.0,failed,37382.000000,Technology,False
...,...,...,...,...,...,...,...
33322,37060,"This book is for, “When Life gets Crappy!” Lik...",500.0,failed,1.000000,Publishing,False
21788,24154,A book of essays on Africa and urban poverty.,575000.0,failed,5.102393,Journalism,False
52221,58290,A poster campaign for the American moment.,86.0,failed,0.000000,Design,False
92761,104716,Telling the story of the city through remarkab...,50000.0,successful,49636.098520,Photography,True


In [35]:
# VADER
def vader_features(text):
    scores = vader.polarity_scores(text)
    return pd.Series({
        "vader_compound": scores["compound"],
        "vader_pos": scores["pos"],
        "vader_neu": scores["neu"],
        "vader_neg": scores["neg"]
    })

col_names = ["vader_compound", "vader_pos", "vader_neu", "vader_neg"]

vader_df = extract_feature(sample_df, vader_features)
logistic_regression_summary(vader_df, col_names)
vader_results = classifier(vader_df, col_names, estimators=[LogisticRegression(), RandomForestClassifier(), KNeighborsClassifier(), SVC()])
vader_results

Optimization terminated successfully.
         Current function value: 0.677763
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 7000
Model:                          Logit   Df Residuals:                     6995
Method:                           MLE   Df Model:                            4
Date:                Fri, 28 Nov 2025   Pseudo R-squ.:                0.001917
Time:                        17:58:09   Log-Likelihood:                -4744.3
converged:                       True   LL-Null:                       -4753.5
Covariance Type:            nonrobust   LLR p-value:                  0.001117
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           -139.3087    121.979     -1.142      0.253    -378.382      99.765
vader_compound   

,Metric,LogisticRegression,RandomForestClassifier,KNeighborsClassifier,SVC
0,success_pred_rate,1.000000,0.699000,0.718667,1.000000
1,accuracy,0.583667,0.540667,0.550333,0.583667
2,precision,0.583667,0.588937,0.593228,0.583667
3,recall,1.000000,0.705311,0.730440,1.000000


In [36]:
def afinn_features(text):
    score = afinn.score(text)
    tokens = nltk.word_tokenize(text.lower())
    
    pos = sum(1 for w in tokens if afinn.score(w) > 0)
    neg = sum(1 for w in tokens if afinn.score(w) < 0)

    return pd.Series({
        "afinn_sum": score,
        "afinn_mean": score / len(tokens) if tokens else 0,
        "afinn_pos_count": pos,
        "afinn_neg_count": neg,
    })

col_names = ["afinn_sum", "afinn_mean", "afinn_pos_count", "afinn_neg_count"]

afinn_df = extract_feature(sample_df, afinn_features)
logistic_regression_summary(afinn_df, col_names)
afinn_results = classifier(afinn_df, col_names, estimators=[LogisticRegression(), RandomForestClassifier(), KNeighborsClassifier(), SVC()])
afinn_results

Optimization terminated successfully.
         Current function value: 0.677152
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 7000
Model:                          Logit   Df Residuals:                     6995
Method:                           MLE   Df Model:                            4
Date:                Fri, 28 Nov 2025   Pseudo R-squ.:                0.002817
Time:                        17:58:41   Log-Likelihood:                -4740.1
converged:                       True   LL-Null:                       -4753.5
Covariance Type:            nonrobust   LLR p-value:                 2.199e-05
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.3855      0.036     10.582      0.000       0.314       0.457
afinn_sum     

,Metric,LogisticRegression,RandomForestClassifier,KNeighborsClassifier,SVC
0,success_pred_rate,0.989333,0.777667,0.759667,0.991333
1,accuracy,0.584333,0.544667,0.534000,0.584333
2,precision,0.584906,0.582512,0.577446,0.584734
3,recall,0.991433,0.776128,0.751571,0.993147


In [37]:
def nrc_features(text):
    emotions = NRCLex(text).raw_emotion_scores

    # Initialize all NRC categories with zero
    categories = [
        'anger','anticipation','disgust','fear','joy',
        'sadness','surprise','trust','positive','negative'
    ]
    out = {f"nrc_{c}": emotions.get(c, 0) for c in categories}

    return pd.Series(out)

col_names = ['nrc_anger','nrc_anticipation','nrc_disgust','nrc_fear','nrc_joy',
        'nrc_sadness','nrc_surprise','nrc_trust','nrc_positive','nrc_negative']

nrc_df = extract_feature(sample_df, nrc_features)
logistic_regression_summary(nrc_df, col_names)
nrc_results = classifier(nrc_df, col_names, estimators=[LogisticRegression(), RandomForestClassifier(), KNeighborsClassifier(), SVC()])
nrc_results

Optimization terminated successfully.
         Current function value: 0.673852
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 7000
Model:                          Logit   Df Residuals:                     6989
Method:                           MLE   Df Model:                           10
Date:                Fri, 28 Nov 2025   Pseudo R-squ.:                0.007677
Time:                        17:59:14   Log-Likelihood:                -4717.0
converged:                       True   LL-Null:                       -4753.5
Covariance Type:            nonrobust   LLR p-value:                 1.172e-11
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.4360      0.037     11.731      0.000       0.363       0.509
nrc_anger  

,Metric,LogisticRegression,RandomForestClassifier,KNeighborsClassifier,SVC
0,success_pred_rate,0.934333,0.804000,0.661000,0.950000
1,accuracy,0.593333,0.569000,0.546000,0.590333
2,precision,0.594720,0.594942,0.598084,0.591579
3,recall,0.952027,0.819532,0.677327,0.962878


In [38]:
el = pd.read_csv("evaluative_lexicon_features.csv")

el_dict = el.set_index("Word").to_dict()

def word_separator(text):
    clear_text = text.translate(str.maketrans("", "", string.punctuation)).lower()
    return clear_text.split()

def el_features(text):
    tokens = word_separator(text)
    vals, exts, emos = [], [], []

    for w in tokens:
        if w in el_dict["Valence"]:
            vals.append(el_dict["Valence"][w])
            exts.append(el_dict["Extremity"][w])
            emos.append(el_dict["Emotionality"][w])

    return pd.Series({
        "el_mean_valence": sum(vals)/len(vals) if vals else 0,
        "el_mean_extremity": sum(exts)/len(exts) if exts else 0,
        "el_mean_emotionality": sum(emos)/len(emos) if emos else 0,
    })

col_names = ["el_mean_valence", "el_mean_extremity", "el_mean_emotionality"]

el_df = extract_feature(sample_df, el_features)
logistic_regression_summary(el_df, col_names)
el_results = classifier(el_df, col_names, estimators=[LogisticRegression(), RandomForestClassifier(), KNeighborsClassifier(), SVC()])
el_results

Optimization terminated successfully.
         Current function value: 0.678158
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 7000
Model:                          Logit   Df Residuals:                     6996
Method:                           MLE   Df Model:                            3
Date:                Fri, 28 Nov 2025   Pseudo R-squ.:                0.001335
Time:                        17:59:57   Log-Likelihood:                -4747.1
converged:                       True   LL-Null:                       -4753.5
Covariance Type:            nonrobust   LLR p-value:                  0.005344
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.3394      0.031     10.852      0.000       0.278       0.401

,Metric,LogisticRegression,RandomForestClassifier,KNeighborsClassifier,SVC
0,success_pred_rate,1.000000,0.869667,0.847000,1.000000
1,accuracy,0.583667,0.581333,0.568667,0.583667
2,precision,0.583667,0.594864,0.589925,0.583667
3,recall,1.000000,0.886351,0.856082,1.000000
